## Overview
In this jupyter notebook, I web scrape and data cleane for NBA team statistics. The focus is on standardizing column names and preparing the data for analysis.

In a previous version, I read in data using the urls directly. Now, I have saved the data as CSV files locally and read them in from there for better reliability.

In [87]:
import pandas as pd

In [102]:
kings_pergame = pd.read_csv('basketball_reference/2425_kings_pergame.csv')
kings_advanced = pd.read_csv('basketball_reference/2425_kings_advanced.csv')

league_pergame = pd.read_csv('basketball_reference/2425_league_pergame.csv')
league_advanced = pd.read_csv('basketball_reference/2425_league_advanced.csv')

In [89]:
# kings_pergame.columns
# Index(['Player', 'Age', 'Pos', 'G', 'GS', 'MP', 'FG', 'FGA', 'FG%', '3P',
#        '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB',
#        'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS'],
#       dtype='object')
# kings_advanced.columns
# Index(['Player', 'Age', 'Pos', 'G', 'GS', 'MP', 'PER', 'TS%', '3PAr', 'FTr',
#        'ORB%', 'DRB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'TOV%', 'USG%', 'OWS',
#        'DWS', 'WS', 'WS/48', 'OBPM', 'DBPM', 'BPM', 'VORP'],
#       dtype='object')
# league_pergame.columns
# Index(['Rk', 'Team', 'G', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P',
#        '2PA', '2P%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL',
#        'BLK', 'TOV', 'PF', 'PTS'],
#       dtype='object')
# league_advanced.columns
# Index(['Rk', 'Team', 'Age', 'W', 'L', 'PW', 'PL', 'MOV', 'SOS', 'SRS', 'ORtg',
#        'DRtg', 'NRtg', 'Pace', 'FTr', '3PAr', 'TS%', 'Offense_eFG%',
#        'Offense_TOV%', 'Offense_ORB%', 'Offense_FT/FGA', 'Defense_eFG%',
#        'Defense_TOV%', 'Defense_DRB%', 'Defense_FT/FGA', 'Arena', 'Attend.',
#        'Attend./G'],
#       dtype='object')

In [90]:
# --------------------------------------------------------------
# 2. Standardize column names (lower, _, pct)
# --------------------------------------------------------------
def clean_cols(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace('%', '_pct', regex=False)
        .str.replace('/', '_', regex=False)
        .str.replace(' ', '_', regex=False)
    )
    return df

kings_pergame = clean_cols(kings_pergame)
kings_advanced = clean_cols(kings_advanced)
league_pergame = clean_cols(league_pergame)
league_advanced = clean_cols(league_advanced)

# KINGS: 'player' for both
kings_pergame = kings_pergame.rename(columns={'player': 'player'})
kings_advanced = kings_advanced.rename(columns={'player': 'player'})

# LEAGUE: 'team' for both (no player in league)
league_pergame = league_pergame.rename(columns={'team': 'team'})
league_advanced = league_advanced.rename(columns={'team': 'team'})

#merge kings
# Drop duplicates from advanced (age, pos, g, gs, mp exist in both)
dup_cols = ['age', 'pos', 'g', 'gs', 'mp']
kings_advanced = kings_advanced.drop(columns=[c for c in dup_cols if c in kings_advanced.columns])

# Unique advanced cols to add
kings_adv_unique = [c for c in kings_advanced.columns if c not in kings_pergame.columns]

kings_merged = kings_pergame.merge(
    kings_advanced[['player'] + kings_adv_unique],
    on='player',
    how='left'
)

# Remove any team totals row if slipped in
kings_merged = kings_merged[~kings_merged['player'].str.contains('Total|Team', na=False)]

print(f"KINGS MERGED: {kings_merged.shape[0]} players, {kings_merged.shape[1]} cols")
print("Kings columns:", kings_merged.columns.tolist())

#merge league
league_adv_unique = [c for c in league_advanced.columns if c not in league_pergame.columns]

league_teams = league_pergame.merge(
    league_advanced[['team'] + league_adv_unique],
    on='team',
    how='left'
)

KINGS MERGED: 20 players, 48 cols
Kings columns: ['player', 'age', 'pos', 'g', 'gs', 'mp', 'fg', 'fga', 'fg_pct', '3p', '3pa', '3p_pct', '2p', '2pa', '2p_pct', 'efg_pct', 'ft', 'fta', 'ft_pct', 'orb', 'drb', 'trb', 'ast', 'stl', 'blk', 'tov', 'pf', 'pts', 'per', 'ts_pct', '3par', 'ftr', 'orb_pct', 'drb_pct', 'trb_pct', 'ast_pct', 'stl_pct', 'blk_pct', 'tov_pct', 'usg_pct', 'ows', 'dws', 'ws', 'ws_48', 'obpm', 'dbpm', 'bpm', 'vorp']


In [91]:
print("\n=== KINGS PLAYER SAMPLE ===")
display(kings_merged.head(3))

print("\n=== LEAGUE TEAM SAMPLE (e.g., Kings) ===")
display(league_teams[league_teams['team'].str.contains('Sacramento Kings', case=False)].T)

print("\nBPM now in Kings roster:")
display(kings_merged[['player', 'bpm', 'obpm', 'dbpm', 'vorp']].head())


=== KINGS PLAYER SAMPLE ===


,player,age,pos,g,gs,mp,fg,fga,fg_pct,3p,...,tov_pct,usg_pct,ows,dws,ws,ws_48,obpm,dbpm,bpm,vorp
0,De'Aaron Fox,26,PG,74,74,35.9,9.7,20.9,0.465,2.9,...,10.1,31.0,3.3,3.2,6.5,0.117,2.6,0.1,2.7,3.2
1,Domantas Sabonis,27,C,82,82,35.7,7.7,13.0,0.594,0.4,...,17.9,22.2,8.6,4.0,12.6,0.206,4.0,2.4,6.5,6.2
2,Keegan Murray,23,SF,77,77,33.6,5.8,12.7,0.454,2.4,...,5.9,18.1,2.6,2.6,5.2,0.096,-0.4,-0.5,-0.9,0.8



=== LEAGUE TEAM SAMPLE (e.g., Kings) ===


,9
rk,10
team,Sacramento Kings
g,82
mp,242.4
fg,43.0
fga,90.1
fg_pct,0.478
3p,12.6
3pa,35.2
3p_pct,0.357



BPM now in Kings roster:


,player,bpm,obpm,dbpm,vorp
0,De'Aaron Fox,2.7,2.6,0.1,3.2
1,Domantas Sabonis,6.5,4.0,2.4,6.2
2,Keegan Murray,-0.9,-0.4,-0.5,0.8
3,Harrison Barnes,-1.8,-0.6,-1.2,0.1
4,Malik Monk,0.6,1.5,-1.0,1.2


In [92]:
league_teams['team'] = league_teams['team'].astype(str).str.replace('*', '', regex=False)
league_teams

,rk,team,g,mp,fg,fga,fg_pct,3p,3pa,3p_pct,...,offense_tov_pct,offense_orb_pct,offense_ft_fga,defense_efg_pct,defense_tov_pct,defense_drb_pct,defense_ft_fga,arena,attend.,attend._g
0,1,Cleveland Cavaliers,82,240.9,44.5,90.8,0.491,15.9,41.5,0.383,...,11.6,25.9,0.187,0.528,12.6,74.8,0.181,Rocket Arena,"796,712","19,432"
1,2,Memphis Grizzlies,82,240.3,44.8,93.3,0.479,13.9,37.9,0.367,...,13.1,28.7,0.196,0.533,12.9,74.9,0.206,FedEx Forum,"683,067","16,660"
2,3,Denver Nuggets,82,242.1,45.4,89.8,0.506,12.0,31.9,0.376,...,12.5,26.7,0.200,0.542,11.3,74.6,0.173,Ball Arena,"811,211","19,786"
3,4,Oklahoma City Thunder,82,240.3,44.6,92.7,0.482,14.5,38.8,0.374,...,10.3,24.2,0.180,0.513,14.9,74.6,0.211,Paycom Center,"754,832","17,973"
4,5,Atlanta Hawks,82,241.2,43.4,91.8,0.472,13.5,37.7,0.358,...,13.2,26.3,0.196,0.560,13.8,76.0,0.202,State Farm Arena,"657,613","16,440"
5,6,Chicago Bulls,82,240.9,43.2,92.0,0.470,15.4,42.0,0.367,...,12.7,22.3,0.173,0.539,10.8,76.6,0.178,United Center,"825,659","20,138"
6,7,Indiana Pacers,82,242.1,43.6,89.3,0.488,13.2,35.8,0.368,...,11.8,21.3,0.190,0.546,13.1,74.5,0.190,Gainbridge Fieldhouse,"685,434","16,737"
7,8,Boston Celtics,82,241.8,41.6,90.0,0.462,17.8,48.2,0.368,...,10.8,25.7,0.169,0.522,11.6,76.0,0.154,TD Garden,"785,396","19,156"
8,9,New York Knicks,82,242.4,43.3,89.2,0.486,12.6,34.1,0.369,...,11.9,26.0,0.186,0.549,13.1,74.5,0.176,Madison Square Garden (IV),"811,794","19,800"
9,10,Sacramento Kings,82,242.4,43.0,90.1,0.478,12.6,35.2,0.357,...,11.8,25.4,0.190,0.557,12.5,76.9,0.200,Golden 1 Center,"696,409","16,986"


In [93]:
# Example metrics list now matches the DataFrame
metrics = ['fg', '3p', 'ft', 'orb', 'drb', 'trb', 'ast', 'stl', 'blk', 'tov', 'pts']
league_teams.columns

Index(['rk', 'team', 'g', 'mp', 'fg', 'fga', 'fg_pct', '3p', '3pa', '3p_pct',
       '2p', '2pa', '2p_pct', 'ft', 'fta', 'ft_pct', 'orb', 'drb', 'trb',
       'ast', 'stl', 'blk', 'tov', 'pf', 'pts', 'age', 'w', 'l', 'pw', 'pl',
       'mov', 'sos', 'srs', 'ortg', 'drtg', 'nrtg', 'pace', 'ftr', '3par',
       'ts_pct', 'offense_efg_pct', 'offense_tov_pct', 'offense_orb_pct',
       'offense_ft_fga', 'defense_efg_pct', 'defense_tov_pct',
       'defense_drb_pct', 'defense_ft_fga', 'arena', 'attend.', 'attend._g'],
      dtype='object')

In [94]:
# Find Sacramento Kings row in league table
kings_team_stats = league_teams[league_teams['team'].str.contains("Sacramento Kings", case=False, na=False)].iloc[0]
kings_team_stats

rk                               10
team               Sacramento Kings
g                                82
mp                            242.4
fg                             43.0
fga                            90.1
fg_pct                        0.478
3p                             12.6
3pa                            35.2
3p_pct                        0.357
2p                             30.5
2pa                            54.9
2p_pct                        0.555
ft                             17.1
fta                            21.2
ft_pct                        0.806
orb                            11.0
drb                            33.2
trb                            44.2
ast                            26.5
stl                             7.6
blk                             4.4
tov                            13.3
pf                             18.9
pts                           115.7
age                            27.8
w                                40
l                           

In [95]:
league_means = league_teams[metrics].mean(numeric_only=True)
league_means

fg      41.686667
3p      13.536667
ft      16.903333
orb     11.123333
drb     32.986667
trb     44.113333
ast     26.546667
stl      8.210000
blk      4.883333
tov     14.306667
pts    113.826667
dtype: float64

In [96]:
# Combine Kings stats and league averages into one DataFrame
comparison = pd.DataFrame({
    'league_avg': league_means,
    'kings': kings_team_stats[league_means.index]  # make sure order matches
})

# Compute differences
comparison['difference'] = comparison['kings'] - comparison['league_avg']

# For metrics where higher is bad (like turnovers), invert the difference
inverse_metrics = ['tov']  # higher turnovers = weakness
comparison['score'] = comparison.apply(
    lambda row: -row['difference'] if row.name not in inverse_metrics else row['difference'],
    axis=1
)

# Normalize to sum to 1 (absolute values)
total = comparison['score'].abs().sum()
comparison['weight'] = (comparison['score'] / total).round(3)

print("\n--- Needs Profile ---\n")
print(comparison[['kings', 'league_avg', 'difference', 'weight']])




--- Needs Profile ---

     kings  league_avg difference  weight
fg    43.0   41.686667   1.313333  -0.191
3p    12.6   13.536667  -0.936667   0.136
ft    17.1   16.903333   0.196667  -0.029
orb   11.0   11.123333  -0.123333   0.018
drb   33.2   32.986667   0.213333  -0.031
trb   44.2   44.113333   0.086667  -0.013
ast   26.5   26.546667  -0.046667   0.007
stl    7.6    8.210000      -0.61   0.089
blk    4.4    4.883333  -0.483333   0.070
tov   13.3   14.306667  -1.006667  -0.146
pts  115.7  113.826667   1.873333  -0.272


In [97]:
league_teams.drop(columns=['arena', 'attend.', 'attend._g'], inplace=True)
league_teams
# kings_merged

,rk,team,g,mp,fg,fga,fg_pct,3p,3pa,3p_pct,...,3par,ts_pct,offense_efg_pct,offense_tov_pct,offense_orb_pct,offense_ft_fga,defense_efg_pct,defense_tov_pct,defense_drb_pct,defense_ft_fga
0,1,Cleveland Cavaliers,82,240.9,44.5,90.8,0.491,15.9,41.5,0.383,...,0.457,0.607,0.578,11.6,25.9,0.187,0.528,12.6,74.8,0.181
1,2,Memphis Grizzlies,82,240.3,44.8,93.3,0.479,13.9,37.9,0.367,...,0.406,0.588,0.554,13.1,28.7,0.196,0.533,12.9,74.9,0.206
2,3,Denver Nuggets,82,242.1,45.4,89.8,0.506,12.0,31.9,0.376,...,0.356,0.604,0.573,12.5,26.7,0.200,0.542,11.3,74.6,0.173
3,4,Oklahoma City Thunder,82,240.3,44.6,92.7,0.482,14.5,38.8,0.374,...,0.419,0.593,0.560,10.3,24.2,0.180,0.513,14.9,74.6,0.211
4,5,Atlanta Hawks,82,241.2,43.4,91.8,0.472,13.5,37.7,0.358,...,0.410,0.579,0.546,13.2,26.3,0.196,0.560,13.8,76.0,0.202
5,6,Chicago Bulls,82,240.9,43.2,92.0,0.470,15.4,42.0,0.367,...,0.457,0.585,0.553,12.7,22.3,0.173,0.539,10.8,76.6,0.178
6,7,Indiana Pacers,82,242.1,43.6,89.3,0.488,13.2,35.8,0.368,...,0.400,0.594,0.562,11.8,21.3,0.190,0.546,13.1,74.5,0.190
7,8,Boston Celtics,82,241.8,41.6,90.0,0.462,17.8,48.2,0.368,...,0.536,0.591,0.561,10.8,25.7,0.169,0.522,11.6,76.0,0.154
8,9,New York Knicks,82,242.4,43.3,89.2,0.486,12.6,34.1,0.369,...,0.382,0.589,0.556,11.9,26.0,0.186,0.549,13.1,74.5,0.176
9,10,Sacramento Kings,82,242.4,43.0,90.1,0.478,12.6,35.2,0.357,...,0.391,0.582,0.548,11.8,25.4,0.190,0.557,12.5,76.9,0.200


In [ ]:
kings_merged.to_csv('2425_kings_merged.csv', index=False)
league_teams.to_csv('2425_league_teams_merged.csv', index=False)